### Imports libraries

In [1]:
from util import get_data_merged, get_data_frames

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
import pickle

from scipy import stats
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.decomposition import PCA
from joblib import dump, load

### Data processing

In [4]:
data_log, data_dash = get_data_frames()
features, labels = get_data_merged(data_log, data_dash)

### Models

In [7]:
def normalized_mae(true_values, predicted_values):
    mae_value = mean_absolute_error(true_values, predicted_values)
    nmae_value = mae_value / np.mean(true_values)
    return nmae_value

In [6]:
def train_random_forest_randomsearch(features: pd.DataFrame, labels: pd.Series):
    X_train, X_val, y_train, y_val = train_test_split(
        features,
        np.ravel(labels),
        test_size=0.20,
        random_state=42
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    rf_model = RandomForestRegressor(random_state=42, n_jobs=2)

    nmae_scorer = make_scorer(normalized_mae, greater_is_better=False)

    kfold = KFold(n_splits=5, shuffle=True, random_state=42)

    param_dist = {
        'n_estimators': [50, 100, 150],
        'max_features': ['auto', 'sqrt'],
        'max_depth': [10, 30, 50, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2],
        'bootstrap': [True, False]
    }

    random_search = RandomizedSearchCV(
        estimator=rf_model,
        param_distributions=param_dist,
        scoring=nmae_scorer,
        cv=kfold,
        n_iter=20, 
        verbose=3,
        n_jobs=2,
        random_state=42
    )

    random_search.fit(X_train_scaled, y_train)

    best_model = random_search.best_estimator_
    best_params = random_search.best_params_

    y_pred = best_model.predict(X_val_scaled)
    mae_value = mean_absolute_error(y_val, y_pred)
    nmae_value = normalized_mae(y_val, y_pred)

    return mae_value, nmae_value, best_model, best_params

In [ ]:
def train_xgboost_randomsearch(features: pd.DataFrame, labels: pd.Series):
    X_train, X_val, y_train, y_val = train_test_split(
        features,
        np.ravel(labels),
        test_size=0.20,
        random_state=42
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=2)

    nmae_scorer = make_scorer(normalized_mae, greater_is_better=False)

    kfold = KFold(n_splits=5, shuffle=True, random_state=42)

    param_dist = {
        'n_estimators': [50, 100, 150],
        'max_depth': [3, 6, 10],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma': [0, 0.1, 0.3]
    }

    random_search = RandomizedSearchCV(
        estimator=xgb_model,
        param_distributions=param_dist,
        scoring=nmae_scorer,
        cv=kfold,
        n_iter=20, 
        verbose=3,
        n_jobs=2,
        random_state=42
    )

    random_search.fit(X_train_scaled, y_train)

    best_model = random_search.best_estimator_
    best_params = random_search.best_params_

    y_pred = best_model.predict(X_val_scaled)
    mae_value = mean_absolute_error(y_val, y_pred)
    nmae_value = normalized_mae(y_val, y_pred)

    return mae_value, nmae_value, best_model, best_params, X_val_scaled, y_val


### Train models

In [ ]:
mae_rf, nmae_rf, rf_model, best_rf_params = train_random_forest_randomsearch(features, labels)
y_pred = best_model.predict(X_val_scaled)
visualize_results(y_val, y_pred, feature_importances, features.columns)
alert_end()

In [ ]:

def visualize_results(y_true, y_pred, feature_importances, feature_names):
    sns.set(style="whitegrid")
    
    plt.figure(figsize=(10, 5))
    plt.scatter(y_true, y_pred, alpha=0.6)
    plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], color='red', linestyle='--')
    plt.title('Comparação entre Valores Reais e Preditos')
    plt.xlabel('Valores Reais')
    plt.ylabel('Valores Preditos')
    plt.show()

    feature_importances_df = pd.DataFrame({
        'Features': feature_names,
        'Importância': feature_importances
    }).sort_values(by='Importância', ascending=False)

    plt.figure(figsize=(10, 5))
    sns.barplot(x='Importância', y='Features', data=feature_importances_df)
    plt.title('Importância das Features')
    plt.xlabel('Importância')
    plt.ylabel('Features')
    plt.show()